# Anytime certification on the JAX fast path (bilinear 2D)

This notebook reproduces the paper's **2D bilinear benchmark** (eq. 39) and shows
the **anytime** behaviour of the certifier: the certified rate is a sound lower
bound at every instant and climbs toward the data-driven ceiling as compute
accumulates.

The field is written with `jax.numpy`, so `verify_stability` automatically uses
the **fused, jitted JAX kernel** (simulation and Theorem-8 test fused, no stored
trajectories). Requires the `[jax]` extra.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyddrv import verify_stability
from pyddrv.systems.fields_jax import bilinear_2d_jax

## 1. The system

$$\dot x = A x + B_1\,[x_1^2,\ x_1 x_2,\ x_2^2]^\top,\qquad
A = \begin{bmatrix} 0 & 2 \\ -1 & -1\end{bmatrix},$$

with `B_1` drawn i.i.d. `N(0, eta)`. The Jacobian is **affine in the state**, so
here the default `L_method="corners"` is *exact* and we keep it. `bilinear_2d_jax`
returns the batched JAX field and its analytic Jacobian.

In [ ]:
f, jac = bilinear_2d_jax(eta=0.3, seed=0)     # JAX field + analytic Jacobian
R, d, tau = 0.7, 2, 5.0

## 2. Certify, recording the anytime trace

We pass the analytic Jacobian (`jac=`) so the corner Lipschitz estimate is exact,
ask for a tight sub-optimality (`delta=0.05`), and cap the run at 60 s.
`record_trace=True` logs `(seconds, alpha, n_cubes)` each refinement round.

In [ ]:
import time
t0 = time.time()
report = verify_stability(f, R=R, d=d, jac=jac, tau=tau, eps=0.01,
                          delta=0.05, max_refine=16, max_seconds=60,
                          record_trace=True)
print(report.summary())
print(f"wall time {time.time()-t0:.1f}s, backend={report.backend}")

The certified rate should land around `alpha >= 0.47`, matching the paper's value
for this system — computed here from **data** (simulated trajectories) rather than
from an SOS program.

## 3. The anytime frontier

`plot_anytime` shows the whole story: `alpha` starts *negative* (a coarse grid
certifies nothing), crosses zero, and rises toward the ceiling (dashed). You can
read off the certified guarantee available at any compute budget.

In [ ]:
from pyddrv.viz import plot_anytime

ax = plot_anytime(report, label="certified rate (anytime)")
ax.set_title("Bilinear 2D: anytime certified decay rate")
plt.show()

In [ ]:
# the raw trace, if you want the numbers
import numpy as np
np.array(report.trace)      # columns: wall-time [s], certified alpha, n_cubes

## 4. The covering grid

The same `pyddrv.viz.plot_stability_2d` grid views work on any 2-D stability
report. For the bilinear system the adaptive refinement is heavier (a tighter
`delta`), so the cube-width and per-cube-rate maps show a rich structure.

In [ ]:
from pyddrv.viz import plot_stability_2d

plot_stability_2d(report, f=f, show_grid=True, grid_color_by="outline")
plt.title("Covering grid over the flow"); plt.show()

plot_stability_2d(report, show_grid=True, grid_color_by="width")
plt.title("Cube widths (log scale)"); plt.show()

## Takeaways

- A `jax.numpy` field gets the fused GPU-style kernel automatically (here on CPU;
  `pip install "pyddrv[jax-cuda]"` targets NVIDIA GPUs with no code change).
- The certified rate is **anytime**: the dashed ceiling is the best the data
  supports, and refinement drives the guarantee toward it.
- `report.suboptimality` quantifies the remaining gap at the binding cube.